# Production Time Series Systems

**Track:** Traditional AI Domains — Time Series  
**Prerequisites:** TS/01-03  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 90–120 minutes

---

### 🔍 Socratic Deep Check
*How do you design a real-time anomaly detection system that processes 100K sensor readings per second and alerts within 5 seconds of an anomaly?*

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors build a forecast model in a notebook and call it done. Seniors design the full system: how does the data arrive (Kafka — NB07)? How are features computed in real-time (Feast — NB08)? How is the model served (FastAPI — NB16)? How do you detect when the model's forecasts degrade (drift detection — NB21)? Production time series is 20% modeling, 80% engineering.

**Mental Model:** A production forecasting system is like a weather station network: sensors collect data continuously (streaming), forecasts are published on schedule (batch), and severe weather alerts trigger immediately (anomaly detection). All three patterns run simultaneously.

**Common Junior Pitfall:** Retraining a forecasting model every day on the full historical dataset. With 3 years × 100K sensors = 100 billion data points, this takes hours. Seniors use online learning (update weights incrementally) or fine-tuning (update on recent data only).

---

In [ ]:
!pip install -q numpy pandas matplotlib scikit-learn

---
## 1 · Anomaly Detection in Time Series

| Method | How | Pros | Cons | Best For |
|--------|-----|------|------|----------|
| **Z-Score** | Flag if \|z\| > 3 | Simple, fast | Assumes normality | Stationary series |
| **IQR** | Flag if outside Q1-1.5×IQR | Robust to outliers | No temporal context | General |
| **Rolling Stats** | Flag if outside rolling μ±3σ | Adapts to trends | Window size sensitive | Drifting series |
| **Isolation Forest** | Anomaly = easy to isolate | Multivariate, non-parametric | Black box | Complex patterns |
| **Forecast Residuals** | Flag if \|actual - predicted\| > threshold | Uses temporal patterns | Needs good model | Forecast-based |

In [ ]:
# Cell 1 — Anomaly detection methods comparison
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
n = 500
t = np.arange(n)
normal = 50 + 10*np.sin(2*np.pi*t/50) + np.random.normal(0, 3, n)

# Inject anomalies
anomaly_indices = [100, 200, 300, 350, 400]
anomalous = normal.copy()
for idx in anomaly_indices:
    anomalous[idx] += np.random.choice([-1, 1]) * np.random.uniform(25, 40)

series = pd.Series(anomalous, name='sensor_reading')

# Method 1: Rolling Z-Score
rolling_mean = series.rolling(30).mean()
rolling_std = series.rolling(30).std()
z_scores = (series - rolling_mean) / rolling_std
z_anomalies = np.abs(z_scores) > 3

# Method 2: Forecast Residual (using simple exponential smoothing)
alpha = 0.3
forecast = [series.iloc[0]]
for i in range(1, len(series)):
    forecast.append(alpha * series.iloc[i-1] + (1-alpha) * forecast[-1])
residuals = series - pd.Series(forecast)
residual_threshold = 3 * residuals.rolling(30).std()
residual_anomalies = np.abs(residuals) > residual_threshold

# Results
print('=== Anomaly Detection Results ===')
print(f'  Injected anomalies at: {anomaly_indices}')
print(f'  Rolling Z-Score found:  {list(z_anomalies[z_anomalies].index.values)}')
print(f'  Forecast Residual found: {list(residual_anomalies[residual_anomalies].index.values)}')

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(series, color='steelblue', linewidth=0.8)
axes[0].scatter(series[z_anomalies].index, series[z_anomalies], color='red', s=60, zorder=5, label='Detected')
axes[0].set_title('Rolling Z-Score Anomaly Detection')
axes[0].legend()
axes[1].plot(np.abs(z_scores), color='orange', linewidth=0.8)
axes[1].axhline(3, color='red', linestyle='--', label='Threshold (z=3)')
axes[1].set_title('Absolute Z-Score Over Time')
axes[1].legend()
plt.tight_layout(); plt.show()

---
## 2 · Production Architecture

### Real-Time Forecasting + Anomaly Detection System

```
┌─────────────┐     ┌──────────┐     ┌───────────────────┐
│ IoT Sensors  │────→│  Kafka   │────→│ Flink/Spark       │
│ (100K/sec)   │     │ (NB07)   │     │ Streaming (NB07)  │
└─────────────┘     └──────────┘     │                   │
                                      │  ┌─────────────┐  │
                                      │  │ Feature Eng  │  │
                                      │  │ (rolling μ,σ)│  │
                                      │  └──────┬──────┘  │
                                      │         ↓         │
                                      │  ┌─────────────┐  │
                                      │  │ Anomaly Det  │──┼──→ PagerDuty Alert
                                      │  │ (z-score)    │  │
                                      │  └─────────────┘  │
                                      └───────────────────┘
                                               ↓
                                      ┌───────────────────┐
                                      │ Feature Store     │
                                      │ (Feast, NB08)     │
                                      └────────┬──────────┘
                                               ↓
                              ┌────────────────────────────────┐
                              │ Batch Forecasting (hourly)     │
                              │ Airflow DAG (NB06b) triggers:  │
                              │   1. Pull features from store  │
                              │   2. Run XGBoost/Chronos       │
                              │   3. Store forecasts in DB     │
                              │   4. Serve via API (NB16)      │
                              └────────────────────────────────┘
```

### Choosing Batch vs Streaming vs Hybrid

| Pattern | Latency | Example | Architecture |
|---------|---------|---------|-------------|
| **Batch** | Hours | Demand forecast for tomorrow | Airflow + XGBoost |
| **Micro-batch** | Minutes | Update every 15 min | Spark Streaming |
| **Streaming** | Seconds | Sensor anomaly detection | Flink + Z-score |
| **Hybrid** | Mixed | Batch forecasts + streaming anomalies | Airflow + Flink |

In [ ]:
# Cell 2 — Simulate a streaming anomaly detection pipeline

import time
from collections import deque

class StreamingAnomalyDetector:
    """Online anomaly detection with rolling statistics.
    Processes one data point at a time (like Flink/Spark Streaming).
    """
    def __init__(self, window_size=50, z_threshold=3.0):
        self.window = deque(maxlen=window_size)
        self.z_threshold = z_threshold
        self.anomaly_count = 0
    
    def process(self, value, timestamp):
        """Process a single data point. Returns alert if anomaly."""
        self.window.append(value)
        
        if len(self.window) < 10:
            return None  # Not enough data yet
        
        arr = np.array(self.window)
        mean, std = arr[:-1].mean(), arr[:-1].std()
        z_score = abs((value - mean) / (std + 1e-10))
        
        if z_score > self.z_threshold:
            self.anomaly_count += 1
            return {
                'timestamp': timestamp,
                'value': round(value, 2),
                'z_score': round(z_score, 2),
                'expected_range': f'[{mean-3*std:.1f}, {mean+3*std:.1f}]',
                'severity': 'CRITICAL' if z_score > 5 else 'WARNING',
            }
        return None

# Simulate streaming
detector = StreamingAnomalyDetector(window_size=50, z_threshold=3.0)

print('=== Streaming Anomaly Detection ===')
print(f'Processing {len(series)} data points...\n')

alerts = []
for i, val in enumerate(series):
    alert = detector.process(val, f't={i}')
    if alert:
        alerts.append(alert)
        print(f'  🚨 {alert["severity"]}: {alert["timestamp"]} | '
              f'value={alert["value"]} | z={alert["z_score"]} | '
              f'expected={alert["expected_range"]}')

print(f'\nTotal alerts: {len(alerts)} | Injected anomalies: {len(anomaly_indices)}')

---
## 3 · Model Selection Decision Tree

```
How many time series do you have?
├── 1 (univariate)
│   ├── < 500 points → ARIMA / ETS / Prophet
│   └── > 500 points → XGBoost with engineered features
├── 10-1000 (related series)
│   ├── Need speed → XGBoost / LightGBM (per-series models)
│   └── Need accuracy → LSTM / TCN (global model)
└── 1000+ (many related series)
    ├── Have compute budget → Train Transformer/N-BEATS
    └── Quick baseline → Foundation model (Chronos, TimesFM)

Anomaly detection?
├── Univariate, stationary → Z-score / IQR
├── Univariate, non-stationary → Forecast residuals
└── Multivariate → Isolation Forest / Autoencoder
```

---
## Knowledge Check

### Q1: Batch vs Streaming
<details><summary>Answer</summary>

**Batch forecasting** runs on a schedule (e.g., hourly via Airflow) and produces forecasts for the next N periods. **Streaming detection** processes each data point as it arrives and triggers alerts within seconds. Most production systems use BOTH: batch for planning, streaming for alerting.
</details>

### Q2: When to choose Chronos over training custom?
<details><summary>Answer</summary>

Use Chronos for quick baselines, limited historical data, or domains where you don't have labeled anomalies. Train custom when you have 1000+ related series with domain-specific patterns (e.g., financial tick data) and strict accuracy requirements.
</details>

---
## Practical Practice

### Exercise 1: System Design
Design a demand forecasting system for an e-commerce platform with 50K products. Specify: data pipeline, feature store, model choice, serving strategy, and monitoring.

### Exercise 2: Anomaly Detector
Extend the `StreamingAnomalyDetector` to handle seasonal patterns (hint: compare against same-time-yesterday instead of rolling window).

---

**Cross-references:** Kafka streaming → NB07 | Feature engineering → NB08 | Pipeline orchestration → NB06b | Model monitoring → NB21 | System observability → NB21b | System design → NB35